# Analiza aktywności użytkowników i treści forum askubuntu.com

Notebook uruchamiany w **EMR Notebook** podłączonym do klastra **Amazon EMR** (Apache Spark).

Kolejność działań:
1. wczytanie plików JSON z bucketu **Amazon S3** i utworzenie Spark DataFrame dla każdej tabeli,
2. sprawdzenie danych,
3. odpowiedzi na 10 pytań analitycznych (a–j) - wynik w postaci tabeli, wykresu lub liczby,
4. zapis wyników w formacie xlsx w buckecie S3 (dane do raportu Power BI).

Wymagane biblioteki instaluje na klastrze skrypt `boto3_install.sh` (bootstrap action).

In [ ]:
import io
import json

import boto3
import matplotlib.pyplot as plt
import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_unixtime, to_date

# Bucket w Amazon S3, w którym znajdują się dane i do którego zapisywane są wyniki
BUCKET_NAME = "bucket1-project-pw"

## 1. Wczytanie danych z Amazon S3

Pliki `all_<tabela>.json` (utworzone skryptem `combine_json.py`) zawierają listę rekordów pobranych z API.
Plik jest pobierany z S3 biblioteką boto3, zamieniany na listę słowników i przekształcany w Spark DataFrame.

API zwraca daty jako *Unix timestamp* (liczba sekund od 01.01.1970), dlatego kolumna `creation_date`
jest zamieniana na zwykłą datę (`yyyy-MM-dd`).

In [ ]:
# Sesja Spark. Dane trafiają do Sparka przez driver (createDataFrame), dlatego zwiększono
# maksymalny rozmiar wiadomości przesyłanej między driverem a executorami (wartość w MB).
spark = SparkSession.builder.appName("askubuntu") \
    .config("spark.rpc.message.maxSize", "1024") \
    .getOrCreate()

s3_client = boto3.client("s3")


def wczytaj_tabele(file_name):
    """Pobiera plik JSON z bucketu S3 i zwraca Spark DataFrame z przekonwertowaną datą utworzenia."""
    # pobranie obiektu z bucketu i konwersja zawartości (bajty -> tekst -> lista słowników)
    object_from_bucket = s3_client.get_object(Bucket=BUCKET_NAME, Key=file_name)
    records = object_from_bucket["Body"].read().decode("utf-8")
    data_list = json.loads(records)

    # utworzenie Spark DataFrame na podstawie listy rekordów
    df = spark.createDataFrame(data_list)

    # konwersja daty: Unix timestamp -> data
    return df.withColumn("creation_date", to_date(from_unixtime(col("creation_date"), "yyyy-MM-dd")))


# Uwaga: na klastrze EMR można też wczytać plik bezpośrednio przez Sparka:
#   spark.read.json(f"s3://{BUCKET_NAME}/all_posts.json")
# wtedy dane nie przechodzą przez driver i są czytane równolegle przez executory.

In [ ]:
df_users = wczytaj_tabele("all_users.json")
df_posts = wczytaj_tabele("all_posts.json")
df_comments = wczytaj_tabele("all_comments.json")
df_questions = wczytaj_tabele("all_questions.json")
df_answers = wczytaj_tabele("all_answers.json")

df_questions.printSchema()

## 2. Sprawdzenie danych

Liczba rekordów, zakres dat oraz liczba pustych wartości w tabeli postów
(posty bez autora to np. posty użytkowników, którzy usunęli konto).

In [ ]:
# liczba rekordów w każdej tabeli
for name, df in [("users", df_users), ("posts", df_posts), ("comments", df_comments),
                 ("questions", df_questions), ("answers", df_answers)]:
    print(f"{name}: {df.count()} rekordów")

# zakres dat postów
df_posts.select(F.min("creation_date").alias("pierwszy_post"),
                F.max("creation_date").alias("ostatni_post")).show()

# liczba pustych wartości w każdej kolumnie tabeli postów
df_posts.select([F.count(F.when(col(c).isNull(), c)).alias(c) for c in df_posts.columns]).show()

## a) Liczba postów na przestrzeni czasu

In [ ]:
# grupowanie postów według roku utworzenia i zliczenie postów w każdym roku
df_posts_per_year = df_posts.groupBy(F.year("creation_date").alias("year")) \
    .agg(F.count("post_id").alias("post_count")) \
    .orderBy("year")

df_posts_per_year.show()

# wynik jest mały (kilkanaście wierszy), więc można go przenieść do pandas i narysować wykres
pandas_df_posts_per_year = df_posts_per_year.toPandas()

plt.figure(figsize=(12, 6))
plt.plot(pandas_df_posts_per_year["year"], pandas_df_posts_per_year["post_count"], marker="o", linestyle="-")
plt.xlabel("Rok")
plt.ylabel("Liczba postów")
plt.title("Liczba postów w kolejnych latach")
plt.grid(True)
plt.show()

## b) Czas na forum - 10 najdłużej aktywnych użytkowników

Czas na forum = liczba dni od założenia konta do ostatniej aktywności (post lub komentarz).
Pominięto konto systemowe *Community* (bot Stack Exchange, `user_id = -1`).

In [ ]:
# aktywność użytkowników: data i autor każdego komentarza oraz każdego posta
df_comments_reduced = df_comments.select(col("creation_date").alias("activity_date"),
                                         col("owner.user_id").alias("user_id"))
df_posts_reduced = df_posts.select(col("creation_date").alias("activity_date"),
                                   col("owner.user_id").alias("user_id"))

# połączenie komentarzy i postów w jeden DataFrame (union łączy wiersze o tych samych kolumnach)
df_comments_posts = df_comments_reduced.union(df_posts_reduced)

# data ostatniej aktywności każdego użytkownika
df_last_activity = df_comments_posts.groupBy("user_id").agg(F.max("activity_date").alias("last_activity"))

# data założenia konta każdego użytkownika
df_users_reduced = df_users.select(col("user_id"), col("creation_date").alias("user_creation_date"))

# połączenie (join) użytkowników z datą ostatniej aktywności i obliczenie liczby dni na forum
df_days_active = df_users_reduced.join(df_last_activity, on="user_id", how="inner") \
    .filter(col("user_id") > 0) \
    .withColumn("days_active", F.datediff(col("last_activity"), col("user_creation_date"))) \
    .orderBy("days_active", ascending=False)

df_days_active.show(10)

# do pandas przenoszone jest tylko 10 wierszy (limit przed toPandas)
pandas_df_days_active = df_days_active.limit(10).toPandas()
pandas_df_days_active.plot.bar(x="user_id", y="days_active", legend=False, figsize=(10, 5))
plt.xlabel("Id użytkownika")
plt.ylabel("Liczba dni na forum")
plt.title("10 najdłużej aktywnych użytkowników")
plt.show()

## c) Porównanie najwyżej i najniżej ocenianych pytań (długość, tagi, liczba odpowiedzi)

Długość pytania liczona jest po usunięciu znaczników HTML z treści (API zwraca treść w formacie HTML).

In [ ]:
remove_tags_regex = r"<[^>]*>"   # znaczniki HTML, np. <p>, </code>
remove_newlines_regex = r"\n"    # znaki nowej linii

# oczyszczenie treści pytania, a następnie obliczenie jej długości i liczby tagów
df_questions_reduced = df_questions.select("question_id", "title", "tags", "body", "answer_count", "score") \
    .withColumn("body", F.regexp_replace(F.regexp_replace(col("body"), remove_tags_regex, ""),
                                         remove_newlines_regex, " ")) \
    .withColumn("question_length", F.length(col("body"))) \
    .withColumn("tag_count", F.size(col("tags")))

columns = ["question_id", "score", "question_length", "tags", "tag_count", "answer_count"]

# 10 najwyżej i 10 najniżej ocenianych pytań
df_top_questions = df_questions_reduced.orderBy("score", ascending=False).limit(10)
df_bottom_questions = df_questions_reduced.orderBy("score", ascending=True).limit(10)

print("Najwyżej oceniane pytania:")
df_top_questions.select(columns).show(truncate=False)
print("Najniżej oceniane pytania:")
df_bottom_questions.select(columns).show(truncate=False)

# porównanie średnich wartości w obu grupach
df_questions_comparison = df_top_questions.withColumn("group", F.lit("najwyżej oceniane")) \
    .union(df_bottom_questions.withColumn("group", F.lit("najniżej oceniane"))) \
    .groupBy("group") \
    .agg(F.round(F.avg("question_length"), 0).alias("avg_length"),
         F.round(F.avg("tag_count"), 2).alias("avg_tag_count"),
         F.round(F.avg("answer_count"), 2).alias("avg_answer_count"))

df_questions_comparison.show()

## d) Procent przypadków, kiedy najwyżej oceniana odpowiedź nie jest odpowiedzią zaakceptowaną

Obliczone są dwie wartości:
* **odsetek pytań bez zaakceptowanej odpowiedzi** - wartość podana w pracy (61,42%),
* **odsetek pytań, w których zaakceptowana odpowiedź ma niższą ocenę niż najlepsza odpowiedź** -
  bezpośrednia odpowiedź na pytanie d) (liczona tylko dla pytań, które mają zaakceptowaną odpowiedź).

In [ ]:
# dla każdego pytania: najwyższa ocena odpowiedzi i informacja, czy którakolwiek odpowiedź została zaakceptowana
df_answers_per_question = df_answers.groupBy("question_id") \
    .agg(F.max("score").alias("max_score"),
         F.max("is_accepted").alias("has_accepted"))   # max z wartości logicznych = True, jeśli choć jedna jest True

# 1) odsetek pytań bez zaakceptowanej odpowiedzi (wartość z pracy)
without_accepted = df_answers_per_question.filter(col("has_accepted") == False).count()
total_questions = df_answers_per_question.count()
pct_without_accepted = without_accepted / total_questions * 100
print(f"Pytania bez zaakceptowanej odpowiedzi: {pct_without_accepted:.2f}%")

# 2) zaakceptowana odpowiedź nie jest najwyżej oceniana
df_accepted = df_answers.filter(col("is_accepted") == True) \
    .select("question_id", col("score").alias("accepted_score"))
df_accepted_vs_max = df_accepted.join(df_answers_per_question, on="question_id")

not_top = df_accepted_vs_max.filter(col("accepted_score") < col("max_score")).count()
pct_accepted_not_top = not_top / df_accepted_vs_max.count() * 100
print(f"Zaakceptowana odpowiedź nie jest najwyżej oceniana: {pct_accepted_not_top:.2f}%")

## e) Rozkład ocen odpowiedzi zaakceptowanych i pozostałych (średnia, odchylenie, minimum, maksimum)

In [ ]:
df_answers_stat = df_answers.select("answer_id", "is_accepted", "score")

# statystyki ocen odpowiedzi zaakceptowanych
df_accepted_stat = df_answers_stat.filter(col("is_accepted") == True).select(
    F.round(F.avg("score"), 2).alias("avg_score"),
    F.round(F.stddev("score"), 2).alias("stddev_score"),
    F.min("score").alias("min_score"),
    F.max("score").alias("max_score")).withColumn("acceptance", F.lit("accepted"))

# statystyki ocen pozostałych odpowiedzi
df_not_accepted_stat = df_answers_stat.filter(col("is_accepted") == False).select(
    F.round(F.avg("score"), 2).alias("avg_score"),
    F.round(F.stddev("score"), 2).alias("stddev_score"),
    F.min("score").alias("min_score"),
    F.max("score").alias("max_score")).withColumn("acceptance", F.lit("not_accepted"))

# połączenie obu wyników w jedną tabelę
df_acc_nacc = df_accepted_stat.union(df_not_accepted_stat) \
    .select("acceptance", "avg_score", "stddev_score", "min_score", "max_score")
df_acc_nacc.show()

## f) Top 5 tagów, które wygenerowały najwięcej wyświetleń

In [ ]:
# explode - każdy tag pytania trafia do osobnego wiersza (z liczbą wyświetleń pytania)
df_questions_tags = df_questions.select(F.explode("tags").alias("tag"), "view_count")

# suma wyświetleń dla każdego tagu i wybór 5 największych
df_top_tags = df_questions_tags.groupBy("tag") \
    .agg(F.sum("view_count").alias("total_views")) \
    .orderBy("total_views", ascending=False) \
    .limit(5)

df_top_tags.show()

## g) Liczba pytań w czasie dla każdego z top 5 tagów

Dla każdego roku liczona jest liczba pytań oznaczonych danym tagiem.
Kropka w nazwie tagu (np. `14.04`) zamieniana jest na podkreślenie, bo kropka w nazwie kolumny Sparka
oznacza odwołanie do pola struktury.

In [ ]:
# lista 5 najpopularniejszych tagów z punktu f)
tags_list = [row["tag"] for row in df_top_tags.collect()]
print(tags_list)

# dla każdego tagu: suma pytań, które zawierają ten tag (when -> 1, w przeciwnym razie 0)
tag_columns = [
    F.sum(F.when(F.array_contains(col("tags"), tag), 1).otherwise(0)).alias(tag.replace(".", "_"))
    for tag in tags_list
]

df_tags_per_year = df_questions.groupBy(F.year("creation_date").alias("year")) \
    .agg(*tag_columns) \
    .orderBy("year")

df_tags_per_year.show()

# wykres - jedna linia dla każdego tagu
pandas_df_tags_per_year = df_tags_per_year.toPandas()

plt.figure(figsize=(10, 6))
for tag in tags_list:
    column = tag.replace(".", "_")
    plt.plot(pandas_df_tags_per_year["year"], pandas_df_tags_per_year[column], marker="o", label=tag)
plt.xlabel("Rok")
plt.ylabel("Liczba pytań")
plt.title("Liczba pytań w kolejnych latach dla top 5 tagów")
plt.legend()
plt.grid(True)
plt.show()

## h) Najczęściej pojawiające się słowa w tytułach (z pominięciem stopwords)

Przetwarzanie tekstu:
1. usunięcie encji HTML (np. `&#39;` = apostrof, `&quot;` = cudzysłów), cyfr i znaków specjalnych,
2. zamiana na małe litery,
3. tokenizacja - podział tytułu na słowa (`RegexTokenizer`),
4. usunięcie *stopwords* - częstych słów bez znaczenia, np. *the, how, to* (`StopWordsRemover`),
5. stemming - sprowadzenie słów do rdzenia, np. *installing, installed → instal* (biblioteka NLTK),
6. zliczenie wystąpień każdego słowa.

In [ ]:
import nltk
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover
from pyspark.sql.types import ArrayType, StringType

df_title = df_questions.select("question_id", "title")

# 1-2. czyszczenie tytułów: encje HTML, cyfry i znaki niebędące literami zamieniane są na spacje
patterns = [r"&#?\w+;",  # encje HTML
            r"\d",       # cyfry
            r"\W"]       # znaki specjalne (wszystko poza literami, cyframi i _)
for pattern in patterns:
    df_title = df_title.withColumn("title", F.regexp_replace("title", pattern, " "))
df_title = df_title.withColumn("title", F.lower("title"))

# 3. tokenizacja (domyślnie podział po białych znakach)
tokenizer = RegexTokenizer(inputCol="title", outputCol="title_words")
df_title_words = tokenizer.transform(df_title)

# 4. usunięcie stopwords (domyślna lista słów angielskich)
sw_remover = StopWordsRemover(inputCol="title_words", outputCol="words_cleaned")
df_title_words = sw_remover.transform(df_title_words)

# 5. stemming - funkcja Pythona uruchamiana na każdym wierszu jako UDF (User Defined Function)
stemmer = nltk.PorterStemmer()


def stem_words(words):
    return [stemmer.stem(word) for word in words]


stem_words_udf = F.udf(stem_words, ArrayType(StringType()))
df_title_words = df_title_words.withColumn("words_stemmed", stem_words_udf(col("words_cleaned")))

# 6. każde słowo w osobnym wierszu (explode) i zliczenie wystąpień
word_counts = df_title_words.select(F.explode("words_stemmed").alias("word")) \
    .groupBy("word").count() \
    .orderBy("count", ascending=False)

word_counts.show(15)

## i) Procent użytkowników, którzy nigdy nic nie opublikowali

Użytkownicy z tabeli `users` łączeni są (*left join*) z liczbą ich postów - użytkownicy bez postów
mają po złączeniu pustą wartość (`null`) w kolumnie `post_count`.

In [ ]:
# liczba postów każdego autora
df_posts_users_agg = df_posts.select(col("owner.user_id").alias("user_id"), col("post_id")) \
    .groupBy("user_id") \
    .agg(F.count("post_id").alias("post_count"))

# left join - zachowuje wszystkich użytkowników, także tych bez postów
df_joined_post_per_user = df_users.select("user_id").join(df_posts_users_agg, on="user_id", how="left")

all_users = df_users.count()
users_without_posts = df_joined_post_per_user.filter(col("post_count").isNull()).count()
percent_users_without_posts = users_without_posts / all_users * 100

print(f"Użytkownicy, którzy nigdy nic nie opublikowali: {percent_users_without_posts:.2f}%")

## j) Średni czas od pojawienia się pytania do pojawienia się zaakceptowanej odpowiedzi

In [ ]:
# data utworzenia pytania
df_question_creation_date = df_questions.select(col("question_id"),
                                                col("creation_date").alias("question_creation_date"))

# data utworzenia zaakceptowanej odpowiedzi
df_answer_accepted_date = df_answers.filter(col("is_accepted") == True) \
    .select(col("question_id"), col("creation_date").alias("answer_creation_date"))

# połączenie pytania z jego zaakceptowaną odpowiedzią i obliczenie różnicy w dniach
df_question_answer_diff = df_question_creation_date.join(df_answer_accepted_date, on="question_id", how="inner") \
    .withColumn("diff", F.datediff(col("answer_creation_date"), col("question_creation_date")))

df_question_answer_diff.show(10)

average_diff = df_question_answer_diff.select(F.mean("diff")).collect()[0][0]
print(f"Średni czas do zaakceptowanej odpowiedzi: {average_diff:.2f} dni")

## 4. Zapis wyników w Amazon S3 (dane do raportu Power BI)

Każdy wynik zapisywany jest jako osobny plik xlsx w buckecie. Pliki pobiera na komputer lokalny skrypt
`3_raport/download_results_from_s3.py`.

In [ ]:
def zapisz_do_s3(pandas_df, file_key):
    """Zapisuje pandas DataFrame jako plik xlsx w buckecie S3 (bez zapisu na dysku klastra)."""
    with io.BytesIO() as output:   # plik tworzony w pamięci
        with pd.ExcelWriter(output, engine="xlsxwriter") as writer:
            pandas_df.to_excel(writer, index=False)
        data = output.getvalue()

    s3 = boto3.resource("s3")
    s3.Bucket(BUCKET_NAME).put_object(Key=file_key, Body=data)
    print(f"Zapisano '{file_key}' w buckecie '{BUCKET_NAME}'")


# wyniki w postaci tabel (Spark DataFrame -> pandas)
zapisz_do_s3(pandas_df_posts_per_year, "wynik_a_posty_w_latach.xlsx")
zapisz_do_s3(pandas_df_days_active, "wynik_b_najdluzej_aktywni.xlsx")
zapisz_do_s3(df_questions_comparison.toPandas(), "wynik_c_porownanie_pytan.xlsx")
zapisz_do_s3(df_acc_nacc.toPandas(), "wynik_e_oceny_odpowiedzi.xlsx")
zapisz_do_s3(df_top_tags.toPandas(), "wynik_f_top_tagi.xlsx")
zapisz_do_s3(pandas_df_tags_per_year, "wynik_g_tagi_w_latach.xlsx")
zapisz_do_s3(word_counts.limit(50).toPandas(), "wynik_h_slowa_w_tytulach.xlsx")

# wyniki liczbowe zebrane w jednej tabeli
df_metrics = pd.DataFrame({
    "metryka": ["d) pytania bez zaakceptowanej odpowiedzi [%]",
                "d) zaakceptowana odpowiedź nie jest najwyżej oceniana [%]",
                "i) użytkownicy bez postów [%]",
                "j) średni czas do zaakceptowanej odpowiedzi [dni]"],
    "wartosc": [round(pct_without_accepted, 2), round(pct_accepted_not_top, 2),
                round(percent_users_without_posts, 2), round(average_diff, 2)],
})
zapisz_do_s3(df_metrics, "wynik_d_i_j_metryki.xlsx")